ベクトルを連結しただけ

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/ConcatTest/ConvNext_{TARGET_EPOCH}epoch_fixed"

# --- 1. モデル定義 (連結方式: Concatenation) ---
class ConcatenationFusionModel(nn.Module):
    def __init__(self, num_classes):
        super(ConcatenationFusionModel, self).__init__()
        
        # 画像特徴抽出: ConvNeXt Tiny (出力は 768チャンネル)
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        img_feature_dim = 768
        hc_feature_dim = 10
        
        # 画像特徴をベクトル化するためのプーリング
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # 分類器: 768 (画像) + 10 (数値) = 778次元を入力とする
        self.classifier = nn.Sequential(
            nn.Linear(img_feature_dim + hc_feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, hc_vec):
        # 1. 画像から特徴抽出: (B, 768, 7, 7)
        x_img = self.dl_extractor(img)
        # 2. プーリングして (B, 768) のベクトルにする
        x_img = self.avgpool(x_img)
        x_img = torch.flatten(x_img, 1)
        
        # 3. 連結 (Concatenation): (B, 768) + (B, 10) -> (B, 778)

        fused = torch.cat([x_img, hc_vec], dim=1)
        
        # 4. 分類
        return self.classifier(fused)

# --- 2. Dataset クラス (変更なし) ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 3. Early Stopping クラス (変更なし) ---
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.delta = delta
        self.path = path

    def __call__(self, val_acc, model):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        if self.verbose:
            print(f'Validation accuracy increased ({self.val_acc_max:.4f} --> {val_acc:.4f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

# --- 4. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 5. 学習実行関数 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    PATIENCE = 20
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_concat_model.pth'

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = ConcatenationFusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path=best_model_path)
    history = {'train_loss': [], 'val_acc': []}

    print(f"Starting Training [Concatenation] on {DEVICE}...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        early_stopping(val_acc, model)
        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {early_stopping.val_acc_max:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Training [Concatenation] on cuda...
Epoch 1/100 | Loss: 1.0273 | Val Acc: 0.5694
Validation accuracy increased (0.0000 --> 0.5694). Saving model...
Epoch 2/100 | Loss: 0.8911 | Val Acc: 0.5278
EarlyStopping counter: 1 out of 20
Epoch 3/100 | Loss: 0.7595 | Val Acc: 0.6181
Validation accuracy increased (0.5694 --> 0.6181). Saving model...
Epoch 4/100 | Loss: 0.6707 | Val Acc: 0.5486
EarlyStopping counter: 1 out of 20
Epoch 5/100 | Loss: 0.5314 | Val Acc: 0.4792
EarlyStopping counter: 2 out of 20
Epoch 6/100 | Loss: 0.4534 | Val Acc: 0.5278
EarlyStopping counter: 3 out of 20
Epoch 7/100 | Loss: 0.3822 | Val Acc: 0.5139
EarlyStopping counter: 4 out of 20
Epoch 8/100 | Loss: 0.2736 | Val Acc: 0.5347
EarlyStopping counter: 5 out of 20
Epoch 9/100 | Loss: 0.2273 | Val Acc: 0.4236
EarlyStopping counter: 6 out of 20
Epoch 10/100 | Loss: 0.2022 | Val Acc: 0.5069
EarlyStopping counter: 7 out of 20
Epoch 11/100 | Loss: 0.1709 | Val Acc: 0.5069
EarlyStopping counter: 8 out of 20
Epoch 12/

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

# 学習時に指定した保存先と重みファイル名
SAVE_DIR = f"../save/ConcatTest/ConvNext_100epoch_fixed"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_concat_model.pth'

# --- 2. モデル定義 (連結方式) ---
class ConcatenationFusionModel(nn.Module):
    def __init__(self, num_classes):
        super(ConcatenationFusionModel, self).__init__()
        
        convnext = models.convnext_tiny(weights=None) # テスト時は重みロードするのでNone
        self.dl_extractor = convnext.features
        img_feature_dim = 768
        hc_feature_dim = 10
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # 連結後の 778次元を入力
        self.classifier = nn.Sequential(
            nn.Linear(img_feature_dim + hc_feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, hc_vec):
        x_img = self.dl_extractor(img)
        x_img = self.avgpool(x_img)
        x_img = torch.flatten(x_img, 1)
        
        # 連結: (B, 778)
        fused = torch.cat([x_img, hc_vec], dim=1)
        return self.classifier(fused)

# --- 3. Dataset クラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # テストデータのスケーリングのために学習データも読み込む
    print("Loading Datasets...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    print(f"Test Data: {len(test_df)} samples")

    # モデル構築と重みロード
    model = ConcatenationFusionModel(num_classes=3).to(DEVICE)
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}")
        return

    state_dict = torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    all_preds, all_labels = [], []

    print(f"Starting Evaluation on {DEVICE}...")

    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 結果表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n=== Test Results ===")
    print(f"Overall Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Concatenation Model)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/test_confusion_matrix.png')
    plt.close()
    print(f"Confusion matrix saved to {SAVE_DIR}/test_confusion_matrix.png")

if __name__ == "__main__":
    evaluate_model()

Loading Datasets...
Test Data: 144 samples
Starting Evaluation on cuda...

=== Test Results ===
Overall Accuracy: 0.5625

Classification Report:
              precision    recall  f1-score   support

           0     0.6444    0.6042    0.6237        48
           1     0.5455    0.5000    0.5217        48
           2     0.5091    0.5833    0.5437        48

    accuracy                         0.5625       144
   macro avg     0.5663    0.5625    0.5630       144
weighted avg     0.5663    0.5625    0.5630       144

Confusion matrix saved to ../save/ConcatTest/ConvNext_100epoch_fixed/test_confusion_matrix.png


10→128次元

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- 1. 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/ConcatExpanded/ConvNext_{TARGET_EPOCH}epoch"

# --- 2. モデル定義 (10 -> 128次元拡張版) ---
class ExpandedConcatFusionModel(nn.Module):
    def __init__(self, num_classes):
        super(ExpandedConcatFusionModel, self).__init__()
        
        # 画像特徴抽出: ConvNeXt Tiny (768次元)
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        self.dl_extractor = convnext.features
        img_feature_dim = 768
        
        # 数値特徴拡張: 10 -> 128次元
        # これにより画像特徴量に対する「声の大きさ」を確保します
        self.hc_upscaler = nn.Sequential(
            nn.Linear(10, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1) # 過学習防止
        )
        hc_expanded_dim = 128
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # 分類器: 768 (画像) + 128 (数値) = 896次元を入力とする
        self.classifier = nn.Sequential(
            nn.Linear(img_feature_dim + hc_expanded_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, hc_vec):
        # 1. 画像特徴 (B, 768)
        x_img = self.dl_extractor(img)
        x_img = self.avgpool(x_img)
        x_img = torch.flatten(x_img, 1)
        
        # 2. 数値特徴拡張 (B, 128)
        x_hc = self.hc_upscaler(hc_vec)
        
        # 3. 連結 (B, 768 + 128 = 896)
        fused = torch.cat([x_img, x_hc], dim=1)
        
        # 4. 分類
        return self.classifier(fused)

# --- 3. Dataset クラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. Early Stopping ---
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, path='checkpoint.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = 0
        self.path = path

    def __call__(self, val_acc, model):
        if self.best_score is None:
            self.best_score = val_acc
            self.save_checkpoint(val_acc, model)
        elif val_acc < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_acc
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        if self.verbose:
            print(f'Validation accuracy increased ({self.val_acc_max:.4f} --> {val_acc:.4f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

# --- 5. 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 6. メイン学習実行 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    PATIENCE = 20
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_expanded_concat_model.pth'

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    val_df = pd.read_csv(VAL_CSV_PATH)
    
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    val_dataset = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = ExpandedConcatFusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path=best_model_path)
    history = {'train_loss': [], 'val_acc': []}

    print(f"Starting Training [10->128 Expanded Concat] on {DEVICE}...")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images, hc_feats)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        val_acc = (correct.double() / total).item()
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        early_stopping(val_acc, model)
        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break
        
        if (epoch + 1) % 5 == 0:
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {early_stopping.val_acc_max:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Starting Training [10->128 Expanded Concat] on cuda...
Epoch 1/100 | Loss: 0.9643 | Val Acc: 0.5972
Validation accuracy increased (0.0000 --> 0.5972). Saving model...
Epoch 2/100 | Loss: 0.8648 | Val Acc: 0.6250
Validation accuracy increased (0.5972 --> 0.6250). Saving model...
Epoch 3/100 | Loss: 0.8002 | Val Acc: 0.5625
EarlyStopping counter: 1 out of 20
Epoch 4/100 | Loss: 0.7622 | Val Acc: 0.4861
EarlyStopping counter: 2 out of 20
Epoch 5/100 | Loss: 0.6117 | Val Acc: 0.5417
EarlyStopping counter: 3 out of 20
Epoch 6/100 | Loss: 0.5312 | Val Acc: 0.5694
EarlyStopping counter: 4 out of 20
Epoch 7/100 | Loss: 0.4066 | Val Acc: 0.4861
EarlyStopping counter: 5 out of 20
Epoch 8/100 | Loss: 0.3236 | Val Acc: 0.5069
EarlyStopping counter: 6 out of 20
Epoch 9/100 | Loss: 0.2944 | Val Acc: 0.5417
EarlyStopping counter: 7 out of 20
Epoch 10/100 | Loss: 0.2420 | Val Acc: 0.5278
EarlyStopping counter: 8 out of 20
Epoch 11/100 | Loss: 0.1835 | Val Acc: 0.4931
EarlyStopping counter: 9 out of 20

In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

# 学習時に指定した保存先と重みファイル名
SAVE_DIR = f"../save/ConcatExpanded/ConvNext_100epoch"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_expanded_concat_model.pth'

# --- 2. モデル定義 (10 -> 128次元拡張版) ---
class ExpandedConcatFusionModel(nn.Module):
    def __init__(self, num_classes):
        super(ExpandedConcatFusionModel, self).__init__()
        
        # 画像特徴抽出: ConvNeXt Tiny
        convnext = models.convnext_tiny(weights=None)
        self.dl_extractor = convnext.features
        img_feature_dim = 768
        
        # 数値特徴拡張: 10 -> 128次元
        self.hc_upscaler = nn.Sequential(
            nn.Linear(10, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        hc_expanded_dim = 128
        
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # 分類器: 768 + 128 = 896次元
        self.classifier = nn.Sequential(
            nn.Linear(img_feature_dim + hc_expanded_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, img, hc_vec):
        x_img = self.dl_extractor(img)
        x_img = self.avgpool(x_img)
        x_img = torch.flatten(x_img, 1)
        
        x_hc = self.hc_upscaler(hc_vec)
        
        fused = torch.cat([x_img, x_hc], dim=1)
        return self.classifier(fused)

# --- 3. Dataset クラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("Loading Datasets for scaling...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    print(f"Test Data: {len(test_df)} samples")

    # モデル構築と重みロード
    model = ExpandedConcatFusionModel(num_classes=3).to(DEVICE)
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}")
        return

    state_dict = torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    all_preds, all_labels = [], []

    print(f"Starting Evaluation on {DEVICE}...")

    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 結果表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n=== Test Results (128-dim Expanded Concat) ===")
    print(f"Overall Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Expanded Concat Model)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/test_confusion_matrix_expanded.png')
    plt.close()
    print(f"Confusion matrix saved to {SAVE_DIR}/test_confusion_matrix_expanded.png")

if __name__ == "__main__":
    evaluate_model()

Loading Datasets for scaling...
Test Data: 144 samples
Starting Evaluation on cuda...

=== Test Results (128-dim Expanded Concat) ===
Overall Accuracy: 0.5694

Classification Report:
              precision    recall  f1-score   support

           0     0.6538    0.7083    0.6800        48
           1     0.6071    0.3542    0.4474        48
           2     0.4844    0.6458    0.5536        48

    accuracy                         0.5694       144
   macro avg     0.5818    0.5694    0.5603       144
weighted avg     0.5818    0.5694    0.5603       144

Confusion matrix saved to ../save/ConcatExpanded/ConvNext_100epoch/test_confusion_matrix_expanded.png
